In [ ]:
!pip install -q langchain-tavily

In [ ]:
# langchain-ollama provides ChatOllama (the only Ollama wrapper with bind_tools).
# arxiv is pinned <4 because LangChain's ArxivAPIWrapper still calls Search.results().
!pip install -q -U langchain langchain-ollama langchain-community langgraph ddgs wikipedia "arxiv<4"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import os

os.environ["TAVILY_API_KEY"] = "tvly-1iHtLV-Dn3IdOQYpTTxPeNGMTRF1K49B9xSp56XrjbwMaZGyU"

In [ ]:
from langchain_ollama import ChatOllama

MODEL_NAME = "qwen3:4b"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url="http://localhost:11434",
    temperature=0.2,
    num_predict=4096,
    request_timeout=120.0,
)

print(f"✅ LLM initialized with model: {llm.model}")

✅ LLM initialized with model: qwen3:4b


In [ ]:
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_tavily import TavilySearch, TavilyExtract

# ── 1. Web search: advanced depth, spam-filtered, agent constrained to legal params ──
web_search_tool = TavilySearch(
    name="web_search",
    max_results=8,
    search_depth="advanced",          # only legal values are "basic" and "advanced"
    include_answer=False,
    include_raw_content=False,        # full pages come from read_url instead
    exclude_domains=[
        "pinterest.com", "quora.com", "facebook.com",
        "instagram.com", "tiktok.com", "medium.com",
    ],
    description=(
        "Semantic web search returning up to 8 relevance-scored results. "
        "REQUIRED: set search_depth to exactly 'advanced' — never pass 'ultra-fast', "
        "'fast', or any other value. Pass include_images as a boolean, not a string. "
        "Prefer keyword-dense queries over full sentences. After searching, call "
        "read_url on the 1-3 highest-scoring URLs before making any factual claim."
    ),
)

# ── 2. Academic search ──────────────────────────────────────────────────────
arxiv_wrapper = ArxivAPIWrapper(top_k_results=3, doc_content_chars_max=2000)
arxiv_tool = ArxivQueryRun(
    api_wrapper=arxiv_wrapper,
    name="arxiv_academic_search",
    description=(
        "Search academic papers on ArXiv. Best for scientific, AI, physics, "
        "math, and CS queries. Returns titles, abstracts, and links."
    ),
)

# ── 3. Wikipedia ────────────────────────────────────────────────────────────
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=2000)
wiki_tool = WikipediaQueryRun(
    api_wrapper=wiki_wrapper,
    name="wikipedia_search",
    description=(
        "Search Wikipedia for definitions, history, background concepts, "
        "and overview knowledge."
    ),
)

# ── 4. read_url:opens 1-3 URLs and returns cleaned full text ──
read_url_tool = TavilyExtract(
    name="read_url",
    extract_depth="advanced",         # "advanced" pulls tables and embedded content
    include_images=False,
    format="markdown",
    description=(
        "Open 1-3 URLs and return their full cleaned page content as markdown. "
        "Use this AFTER web_search or arxiv_academic_search to read the most "
        "promising sources in full instead of trusting snippets. Report only "
        "what the returned text contains; never invent figures, dates, or quotes."
    ),
)

tools = [web_search_tool, arxiv_tool, wiki_tool, read_url_tool]
print(f"✅ {len(tools)} tools configured: {[t.name for t in tools]}")

/tmp/ipykernel_1332/3103834848.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun


✅ 4 tools configured: ['web_search', 'arxiv_academic_search', 'wikipedia_search', 'read_url']


In [ ]:
SYSTEM_PROMPT = """You are an expert, meticulous Senior Research Agent.
Your goal is to conduct comprehensive, objective, and well-supported research.

Follow this systematic methodology:
1. **Understand & Deconstruct**: Break complex queries into sub-questions.
2. **Tool Selection**:
   - `wikipedia_search` — definitions, history, foundational concepts.
   - `arxiv_academic_search` — state-of-the-art papers, formal studies, citations.
   - `web_search` — recent news, benchmarks, implementations. Always use
     search_depth='advanced'.
   - `read_url` — after searching, open the 1-3 highest-scoring URLs in full
     before making any factual claim.
3. **Search → Read → Write**: Never answer a technical question from a single
   snippet. Search first, read the best sources, then synthesize.
4. **Verify & Attribute**: Every claim in the final report must trace to a URL
   returned by a tool. If no retrieved source supports a claim, say so.
5. **Structured Delivery**:
   - **Executive Summary**
   - **Key Findings & Analysis**
   - **Supporting Evidence / Academic Insights**
   - **References & Sources Consulted**

Be thorough, precise, and analytical. Never hallucinate facts, dates, or citations."""

print("✅ System prompt configured!")

✅ System prompt configured!


In [ ]:
from langchain.agents import create_agent

research_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Research Agent built successfully with 4 tools!")

✅ Research Agent built successfully with 4 tools!


In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.errors import GraphRecursionError
from IPython.display import display, Markdown

MAX_STEPS = 50  # 4 tools and search→read→write cycles need more headroom than 40

def run_research(query: str):
    """Runs the research agent and visualizes its reasoning stream."""
    print("=" * 70)
    print(f"🔬 RESEARCH QUERY: {query}")
    print("=" * 70 + "\n")

    inputs = {"messages": [HumanMessage(content=query)]}
    config = {"recursion_limit": MAX_STEPS}
    final_response = ""
    seen = 0

    try:
        for chunk in research_agent.stream(inputs, config=config, stream_mode="values"):
            new_messages = chunk["messages"][seen:]
            seen = len(chunk["messages"])
            for msg in new_messages:
                if msg.type == "ai":
                    for tc in getattr(msg, "tool_calls", []) or []:
                        print(f"🛠️  [Agent Action] Calling `{tc['name']}` "
                              f"with args: {tc['args']}")
                    if msg.content and not getattr(msg, "tool_calls", None):
                        final_response = msg.content
                elif msg.type == "tool":
                    status = ("❌ error" if getattr(msg, "status", "") == "error"
                              else f"{len(msg.content)} chars received")
                    print(f"📥 [Tool Result: {msg.name}] ({status})")
    except GraphRecursionError:
        print(f"\n⚠️ Hit the {MAX_STEPS}-step limit. Try a narrower question "
              f"or raise MAX_STEPS.")

    print("\n" + "=" * 70)
    print("📋 FINAL RESEARCH REPORT")
    print("=" * 70)
    if final_response:
        display(Markdown(final_response))
    else:
        print("(No final report was produced.)")
    return final_response

In [ ]:
# 1. Install zstd (Ollama's installer needs it for decompression)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, socket, time, urllib.request

def _server_running(host="127.0.0.1", port=11434):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex((host, port)) == 0

def _wait_for_ollama(timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            return True
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama server did not come up in time")

# 3. Start the server (idempotent — safe to re-run)
if not _server_running():
    print("Starting Ollama server...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
_wait_for_ollama()

# 4. Pull the model and verify
assert "MODEL_NAME" in globals(), "Run the LLM-config cell before this one."
print(f"Pulling model: {MODEL_NAME} ...")
!ollama pull {MODEL_NAME}
!ollama list

# 5. Run the research
topic = (
    "What are recent advances in Test-Time Compute and Reasoning Models "
    "(e.g. DeepSeek-R1, OpenAI o-series)?"
)
report = run_research(topic)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loca

In [ ]:
run_research("summarize the latest quantum computing news")